In [ ]:
# First, install required packages if not already installed
# pip install numpy pandas xgboost scikit-learn scipy

import numpy as np
import pandas as pd
try:
    import xgboost as xgb
except ImportError:
    raise ImportError("Please install xgboost using: pip install xgboost")
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from scipy.optimize import differential_evolution
import warnings
warnings.filterwarnings('ignore')

def round_weights(weights):
    """Round weights according to specifications"""
    # Round w1, w2, w3 to 2 decimal places
    w1, w2, w3 = np.round(weights[:3], 2)
    # Round wPL to 7 decimal places (for 0.0001 precision)
    wPL = np.round(weights[3], 7)
    return np.array([w1, w2, w3, wPL])

def prepare_data(data):
    """Prepare data for modeling"""
    # Convert DC to numerical values
    le = LabelEncoder()
    data['DC_encoded'] = le.fit_transform(data['DC'])
    
    # Split features and target
    X = data[['w1', 'w2', 'w3', 'wPL', 'DC_encoded']]
    y = data['range']
    
    return X, y, le

def train_optimized_model(X, y):
    """Train XGBoost model with hyperparameter tuning"""
    try:
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        # Define parameter space
        param_space = {
            'max_depth': [3, 4, 5, 6, 7],
            'learning_rate': [0.01, 0.05, 0.1],
            'n_estimators': [100, 200, 300],
            'min_child_weight': [1, 3, 5],
            'subsample': [0.8, 0.9, 1.0],
            'colsample_bytree': [0.8, 0.9, 1.0]
        }
        
        # Initialize model
        model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
        
        # Perform random search
        random_search = RandomizedSearchCV(
            model, param_space, n_iter=20, cv=5, 
            scoring='neg_mean_squared_error', random_state=42
        )
        
        # Fit model
        random_search.fit(X_train, y_train)
        
        # Get best model
        best_model = random_search.best_estimator_
        
        # Print model performance
        train_score = best_model.score(X_train, y_train)
        test_score = best_model.score(X_test, y_test)
        print(f"Train R² Score: {train_score:.4f}")
        print(f"Test R² Score: {test_score:.4f}")
        
        return best_model
    
    except Exception as e:
        print(f"Error during model training: {str(e)}")
        raise

def objective_function(weights, model, dc_encoded):
    """Objective function for optimization"""
    # Round weights to specified precision
    weights = round_weights(weights)
    
    # Combine weights with DC
    X = np.append(weights, dc_encoded)
    X = X.reshape(1, -1)
    
    # Predict range
    range_pred = model.predict(X)[0]
    
    # Return negative range (since we want to maximize)
    return -range_pred

def find_optimal_weights(model, dc_encoded):
    """Find optimal weights using differential evolution"""
    # Define bounds: (0,1) for w1,w2,w3 and (0,0.0001) for wPL
    bounds = [(0, 1), (0, 1), (0, 1), (0, 0.0001)]
    
    result = differential_evolution(
        objective_function,
        bounds,
        args=(model, dc_encoded),
        maxiter=100,
        popsize=20,
        seed=42
    )
    
    # Round the optimal weights
    optimal_weights = round_weights(result.x)
    return optimal_weights, -result.fun

def optimize_all_dcs(data, model, le):
    """Find optimal weights for each drive cycle"""
    results = []
    
    for dc in data['DC'].unique():
        dc_encoded = le.transform([dc])[0]
        optimal_weights, max_range = find_optimal_weights(model, dc_encoded)
        
        results.append({
            'DC': dc,
            'w1': optimal_weights[0],
            'w2': optimal_weights[1],
            'w3': optimal_weights[2],
            'wPL': optimal_weights[3],
            'predicted_range': max_range
        })
    
    return pd.DataFrame(results)

def find_optimal_weights_for_dcs(data):
    """Main function to find optimal weights for all drive cycles"""
    try:
        # Prepare data
        X, y, le = prepare_data(data)
        
        # Train model
        print("Training model...")
        model = train_optimized_model(X, y)
        
        # Find optimal weights for each DC
        print("\nFinding optimal weights for each drive cycle...")
        results = optimize_all_dcs(data, model, le)
        
        return results, model
    
    except Exception as e:
        print(f"Error in optimization process: {str(e)}")
        raise

def predict_range(model, w1, w2, w3, wPL, dc, le):
    """Predict range for given weights and drive cycle"""
    # Round inputs to specified precision
    w1, w2, w3 = np.round([w1, w2, w3], 2)
    wPL = np.round(wPL, 7)
    
    dc_encoded = le.transform([dc])[0]
    X = np.array([[w1, w2, w3, wPL, dc_encoded]])
    return model.predict(X)[0]

In [4]:
# Read your data
data = pd.read_csv('../Final_DataPoints_With_DC.csv')

# Find optimal weights
results, trained_model = find_optimal_weights_for_dcs(data)

# Display results
print("\nOptimal weights for each drive cycle:")
print(results)

Training model...
Train R² Score: 0.9792
Test R² Score: 0.6479

Finding optimal weights for each drive cycle...

Optimal weights for each drive cycle:
                  DC    w1    w2    w3       wPL  predicted_range
0      NEDC_extended  0.04  0.75  0.15  0.000055       202.744949
1      FHDS_extended  0.34  0.94  0.12  0.000071       228.335724
2    US06HW_extended  0.94  0.60  0.95  0.000097       182.773636
3  US06City_extended  0.04  0.26  0.15  0.000056       183.935608
4  FUDSfast_extended  0.33  0.95  0.12  0.000019       166.193192
5      US06_extended  0.94  0.51  0.40  0.000020       188.041992
6      FUDS_extended  0.39  0.75  0.24  0.000072       196.215225


In [5]:
# savig results to a csv file
results.to_csv('XGBOOST_Optimal_Weights.csv', index=False)